<a href="https://colab.research.google.com/github/Ganasa18/belajar-tensorflow/blob/main/local_tts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1 Mount Google
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

ROOT = Path("/content/drive/MyDrive/jarwo_tts")

DATASET_DIR = ROOT / "video_dataset"
WAV_DIR = DATASET_DIR / "wavs"
METADATA = DATASET_DIR / "metadata.csv"

EXPORT_DIR = ROOT / "export"
BACKUP_DIR = ROOT / "checkpoints"

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

print("Metadata exists :", METADATA.exists())
print("WAV count       :", len(list(WAV_DIR.glob("*.wav"))))

Mounted at /content/drive
Metadata exists : True
WAV count       : 17


In [2]:
# 2. INSTALL FFMPEG WHISPER
%cd /content

!apt-get update -qq
!apt-get install -y -qq \
    build-essential \
    cmake \
    ninja-build \
    git \
    espeak-ng

!rm -rf piper1-gpl

!git clone --depth 1 \
    https://github.com/OHF-Voice/piper1-gpl.git

%cd /content/piper1-gpl

!pip install -q -e ".[train]"

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 85.3 MB/s eta 0:00:00


In [ ]:
!chmod +x build_monotonic_align.sh
!./build_monotonic_align.sh

!python3 setup.py build_ext --inplace

In [3]:
# 3 Check GPU

import torch

print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU tidak aktif")

PyTorch : 2.11.0+cpu
CUDA    : False


In [4]:
# 4 — Check Audio

print(open(METADATA, encoding="utf-8").read())

000001.wav|Di tempat kulahir dan dibesarkan, di kota yang dekat laut ini, setelah
000002.wav|segi nama aku pulang, sudah ada shopping mall berdiri.
000003.wav|Waktu itu selalu seperti tongkacir, walaupun telah mengubah pemandangan suara
000004.wav|ombak dan aroma kewombang masih sama seperti dulu.
000005.wav|Saya berhenti sekarang juga, kamu yang terlis di mewah, ada di pojok
000006.wav|renang bukutahunan kita, sumbuk memang kamu yang terlis di mewah.
000007.wav|Berapa kali aku buka untuk memastikannya?
000008.wav|Kamu berdiri di kasir counter, kita cita kamu menjadi seorang hastylist.
000009.wav|Waktu itu kamu pernah bercerita, walaupun seperti yang kamu bayangkan kamu terlihat
000010.wav|Aku jadi legaku dengan gawau kamu sudah menikah, aku terlambat
000011.wav|bilang suka kepadamu, aku dengar kamu pun sekarang punya anak.
000012.wav|Tak sambung memanggilmu fredor masamudahku.
000013.wav|Sekarang juga kamu yang terlis di mewah, ada di pojok renang bukutahunan kita,
000014.wav|sumbuk m

In [ ]:
!pip install -q huggingface_hub

In [ ]:
# 5. Download Pretrained
from huggingface_hub import hf_hub_download
from pathlib import Path

BASE_DIR = Path("/content/base_piper")
BASE_DIR.mkdir(parents=True, exist_ok=True)

BASE_CKPT = hf_hub_download(
    repo_id="rhasspy/piper-checkpoints",
    repo_type="dataset",
    filename=(
        "id/id_ID/news_tts/medium/"
        "epoch=4927-step=232092.ckpt"
    ),
    local_dir=str(BASE_DIR)
)

print(BASE_CKPT)

In [ ]:
# 6. Copy dataset Drive → storage Colab
import shutil
from pathlib import Path

LOCAL_DATASET = Path("/content/jarwo_dataset")
LOCAL_WAV = LOCAL_DATASET / "wavs"
LOCAL_METADATA = LOCAL_DATASET / "metadata.csv"

if LOCAL_DATASET.exists():
    shutil.rmtree(LOCAL_DATASET)

LOCAL_WAV.mkdir(parents=True, exist_ok=True)

for wav in WAV_DIR.glob("*.wav"):
    shutil.copy2(
        wav,
        LOCAL_WAV / wav.name
    )

shutil.copy2(
    METADATA,
    LOCAL_METADATA
)

print("WAV copied:", len(list(LOCAL_WAV.glob("*.wav"))))
print("Metadata:", LOCAL_METADATA)

In [ ]:
# 7. Tentukan lokasi config dan training

CONFIG_FILE = ROOT / "jarwo_voice.json"

RUN_DIR = Path("/content/jarwo_training")
CACHE_DIR = Path("/content/jarwo_cache")

print("Config:", CONFIG_FILE)

In [ ]:
# 8. TRAINING
import subprocess
import shutil

if RUN_DIR.exists():
    shutil.rmtree(RUN_DIR)

if CACHE_DIR.exists():
    shutil.rmtree(CACHE_DIR)

RUN_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    "python3",
    "-m",
    "piper.train",
    "fit",

    "--data.voice_name",
    "jarwo_voice",

    "--data.csv_path",
    str(LOCAL_METADATA),

    "--data.audio_dir",
    str(LOCAL_WAV),

    "--model.sample_rate",
    "22050",

    "--data.espeak_voice",
    "id",

    "--data.cache_dir",
    str(CACHE_DIR),

    "--data.config_path",
    str(CONFIG_FILE),

    "--data.batch_size",
    "2",

    "--data.validation_split",
    "0.10",

    "--data.num_test_examples",
    "1",

    "--data.num_workers",
    "2",

    "--model.warmstart_ckpt",
    str(BASE_CKPT),

    "--model.mos_metric",
    "none",

    "--trainer.accelerator",
    "gpu",

    "--trainer.devices",
    "1",

    "--trainer.default_root_dir",
    str(RUN_DIR),

    "--trainer.max_steps",
    "500",

    "--trainer.max_epochs",
    "500",

    "--trainer.log_every_n_steps",
    "5",
]

print("=== START TRAINING ===")

subprocess.run(
    cmd,
    cwd="/content/piper1-gpl",
    check=True
)

In [ ]:
# 9. Cari checkpoint
from pathlib import Path

ckpts = list(
    RUN_DIR.rglob("*.ckpt")
)

print("Checkpoint ditemukan:", len(ckpts))

for ckpt in ckpts:
    print(ckpt)

In [ ]:
# 10. Backup checkpoint ke Drive
import shutil

DRIVE_CKPT = (
    CHECKPOINT_DIR /
    "jarwo_500steps.ckpt"
)

shutil.copy2(
    FINAL_CKPT,
    DRIVE_CKPT
)

print("Backup ✅")
print(DRIVE_CKPT)

In [ ]:
# 11. Export menjadi ONNX
ONNX_FILE = (
    EXPORT_DIR /
    "jarwo_test.onnx"
)

subprocess.run(
    [
        "python3",
        "-m",
        "piper.train.export_onnx",

        "--checkpoint",
        str(FINAL_CKPT),

        "--output-file",
        str(ONNX_FILE),
    ],
    cwd="/content/piper1-gpl",
    check=True
)

print("ONNX:")
print(ONNX_FILE)

In [ ]:
# 12. Copy config
from pathlib import Path
import shutil

ONNX_CONFIG = Path(
    str(ONNX_FILE) + ".json"
)

shutil.copy2(
    CONFIG_FILE,
    ONNX_CONFIG
)

print(ONNX_FILE)
print(ONNX_CONFIG)

In [ ]:
# 13. Test suara
TEST_TEXT = (
    "Halo, sekarang saya sedang mencoba suara baru. "
    "Baterai perangkat masih enam puluh lima persen."
)

TEST_WAV = (
    EXPORT_DIR /
    "test_voice.wav"
)

In [ ]:
result = subprocess.run(
    [
        "python3",
        "-m",
        "piper",

        "-m",
        str(ONNX_FILE),

        "-c",
        str(ONNX_CONFIG),

        "-f",
        str(TEST_WAV),

        "--",
        TEST_TEXT,
    ],
    cwd="/content/piper1-gpl",
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("TTS gagal")

print(TEST_WAV)

In [ ]:
from IPython.display import Audio, display

display(
    Audio(
        str(TEST_WAV)
    )
)